# 04 SHAP Comparison

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv('../dataset/pd_speech_features.csv', header=1)
X = df.drop(['id', 'class'], axis=1)
y = df['class']
groups = df['id']


In [2]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Full preprocessing
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)
vt_cols = X.columns[vt.get_support()]

scaler = StandardScaler()
X_sc = scaler.fit_transform(X_vt)

skb = SelectKBest(f_classif, k=50)
X_skb = skb.fit_transform(X_sc, y)
selected_features = vt_cols[skb.get_support()]

X_final = pd.DataFrame(X_skb, columns=selected_features)

smote = SMOTE(random_state=RANDOM_STATE)
X_res, y_res = smote.fit_resample(X_final, y)

models = {
    'XGBoost': xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE),
    'LightGBM': lgb.LGBMClassifier(verbose=-1, random_state=RANDOM_STATE),
    'CatBoost': cb.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
}

top10_features = {}

for name, model in models.items():
    model.fit(X_res, y_res)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_final)
    
    # Handle CatBoost/LightGBM which might return list for multi-class/binary format
    if isinstance(shap_values, list):
        shap_values = shap_values[1] # Take positive class
        
    plt.figure()
    shap.summary_plot(shap_values, X_final, max_display=10, plot_type="bar", show=False)
    plt.title(f"{name} Top 10 SHAP")
    plt.savefig(f"{name.lower()}_shap_bar.png", bbox_inches='tight')
    plt.close()
    
    # Calculate top 10 mean abs SHAP
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top10_idx = np.argsort(mean_abs_shap)[-10:]
    top10_features[name] = set(selected_features[top10_idx])

# Compute overlap
overlap_xgb_lgb = len(top10_features['XGBoost'].intersection(top10_features['LightGBM']))
overlap_xgb_cb = len(top10_features['XGBoost'].intersection(top10_features['CatBoost']))
overlap_lgb_cb = len(top10_features['LightGBM'].intersection(top10_features['CatBoost']))
overlap_all = len(top10_features['XGBoost'].intersection(top10_features['LightGBM']).intersection(top10_features['CatBoost']))

print("SHAP Top-10 Overlaps:")
print(f"XGB-LGB: {overlap_xgb_lgb}")
print(f"XGB-CB: {overlap_xgb_cb}")
print(f"LGB-CB: {overlap_lgb_cb}")
print(f"All 3: {overlap_all}")

with open('../results/shap_overlap.txt', 'w') as f:
    f.write(f"XGB-LGB:{overlap_xgb_lgb}\nXGB-CB:{overlap_xgb_cb}\nLGB-CB:{overlap_lgb_cb}\nAll:{overlap_all}\n")


SHAP Top-10 Overlaps:
XGB-LGB: 8
XGB-CB: 9
LGB-CB: 8
All 3: 8
